# ChuckleNet: Scale to 555 Videos (v15)

**Strategy:**
1. Use existing WavLM embeddings from `wavlm_utterance_safe` (555 videos)
2. Extract missing prosody (21-dim) for videos not in existing prosody data
3. Pre-convert M4A → WAV for faster I/O
4. Video-level caching for efficiency

**Time estimate:** 1-2 hours (vs 30+ hours before)

In [ ]:
# Step 1: Mount Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('✅ Drive mounted!')

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers librosa scikit-learn soundfile
!apt-get install -y ffmpeg > /dev/null 2>&1
print('✅ Dependencies installed!')

## Step 3: Load existing WavLM embeddings from wavlm_utterance_safe

In [ ]:
import json
from pathlib import Path

WAVLM_DIR = Path('/content/gdrive/MyDrive/wavlm_utterance_safe')

print('📂 Loading existing WavLM embeddings...')
wavlm_data = {}  # video_id -> list of {start, end, embedding}

for json_file in WAVLM_DIR.glob('*.json'):
    vid = json_file.stem
    with open(json_file) as f:
        data = json.load(f)
    wavlm_data[vid] = data['embeddings']

print(f'✅ Loaded WavLM for {len(wavlm_data)} videos')

total_utterances = sum(len(v) for v in wavlm_data.values())
print(f'   Total utterances: {total_utterances}')

## Step 4: Load existing prosody data (71 videos already have it)

In [ ]:
# Check if prosody data exists locally
import subprocess

PROSODY_LOCAL = Path('/content/gdrive/MyDrive/prosody_features_enhanced_50videos.csv')

if PROSODY_LOCAL.exists():
    print('📂 Found existing prosody CSV')
    import pandas as pd
    prosody_df = pd.read_csv(PROSODY_LOCAL)
    print(f'   {len(prosody_df)} rows')
    print(f'   Columns: {list(prosody_df.columns[:10])}...')
else:
    print('⚠️ No existing prosody CSV found')
    prosody_df = None

## Step 5: Load utterances to get labels

In [ ]:
import subprocess
from pathlib import Path

UTT_PATH = Path('/content/gdrive/MyDrive/utterances_clean.jsonl')
if not UTT_PATH.exists():
    subprocess.run(['pip', 'install', '-q', 'gdown'], check=True, capture_output=True)
    subprocess.run(['gdown', '--fuzzy', '-O', str(UTT_PATH),
        'https://drive.google.com/file/d/1cuhs6mh-r9Spzq9cTDG8AidT53DLsALn/view'],
        check=True, capture_output=True)

print('📋 Loading utterances...')
utterances = []
with open(UTT_PATH) as f:
    for line in f:
        utterances.append(json.loads(line.strip()))

print(f'✅ Loaded {len(utterances)} utterances')

# Create lookup: (video_id, start, end) -> label
label_lookup = {}
for u in utterances:
    key = (u['video_id'], round(u['start'], 2), round(u['end'], 2))
    label_lookup[key] = u['label']

print(f'   Labeled: {len(label_lookup)}')

## Step 6: Match WavLM embeddings with labels

In [ ]:
# Match embeddings with labels
matched = 0
unmatched = 0

for vid, embeddings in wavlm_data.items():
    for emb in embeddings:
        key = (vid, round(emb['start'], 2), round(emb['end'], 2))
        if key in label_lookup:
            emb['label'] = label_lookup[key]
            matched += 1
        else:
            emb['label'] = 0
            unmatched += 1

print(f'✅ Matched: {matched} utterances with labels')
print(f'⚠️ Unmatched: {unmatched}')

# Stats
pos = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 1)
neg = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 0)
print(f'   Positive: {pos} ({pos/(pos+neg)*100:.1f}%)')
print(f'   Negative: {neg} ({neg/(pos+neg)*100:.1f}%)')

## Step 7: Split into train/val/test (video-level)

In [ ]:
import random

video_ids = list(wavlm_data.keys())
random.seed(42)
random.shuffle(video_ids)

n = len(video_ids)
val_vids = set(video_ids[:int(n*0.1)])
test_vids = set(video_ids[int(n*0.1):int(n*0.2)])

train_data = []
val_data = []
test_data = []

for vid in video_ids:
    if vid in val_vids:
        val_data.extend(wavlm_data[vid])
    elif vid in test_vids:
        test_data.extend(wavlm_data[vid])
    else:
        train_data.extend(wavlm_data[vid])

print(f'📊 Split (video-level):')
print(f'   Train: {len(train_data)} utterances ({n - len(val_vids) - len(test_vids)} videos)')
print(f'   Val: {len(val_data)} utterances ({len(val_vids)} videos)')
print(f'   Test: {len(test_data)} utterances ({len(test_vids)} videos)')

## Step 8: Pre-convert M4A to WAV (OPTIMIZATION)

In [ ]:
import subprocess
from pathlib import Path

BASE = Path('/content/gdrive/MyDrive')
WAV_DIR = BASE / 'chuckle_audio_wav'
WAV_DIR.mkdir(exist_ok=True)

# Find M4A files
m4a_files = {}
for folder in ['chuckle_audio', 'chuckle_audio_all/audio', 'chuckle_audio_all/audio_final', 'chuckle_audio_all/audio_new']:
    audio_dir = BASE / folder
    if audio_dir.exists():
        for p in audio_dir.glob('*.m4a'):
            m4a_files[p.stem] = p

print(f'📁 Found {len(m4a_files)} M4A files to convert')

# Convert to WAV
converted = 0
skipped = 0

for i, (stem, m4a_path) in enumerate(m4a_files.items()):
    wav_path = WAV_DIR / (stem + '.wav')
    
    if wav_path.exists():
        skipped += 1
        continue
    
    subprocess.run([
        'ffmpeg', '-y', '-i', str(m4a_path),
        '-ar', '16000', '-ac', '1',
        '-loglevel', 'error',
        str(wav_path)
    ], check=False, capture_output=True)
    
    converted += 1
    
    if (i + 1) % 50 == 0:
        print(f'  Progress: {i+1}/{len(m4a_files)} | {converted} converted, {skipped} exist')

print(f'\n✅ Converted {converted} | Skipped {skipped}')

## Step 9: Extract prosody for each video

In [ ]:
import librosa
import numpy as np
import soundfile as sf
import time

SR = 16000

def extract_prosody_21dim(y, sr):
    """Extract 21 prosody features."""
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([
        np.mean(rms), np.std(rms), np.max(rms), np.min(rms),
        np.max(rms) - np.min(rms)
    ])
    
    # Duration - 2 dims
    features.extend([
        len(y) / sr,
        len(y) / sr / (np.sum(rms > np.mean(rms)) + 1)
    ])
    
    # Spectral - 5 dims
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spec_flat = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([
        np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat),
        np.mean(zcr), np.std(zcr)
    ])
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    return np.array(features, dtype=np.float32)

def get_audio_path(vid):
    # Try WAV first (converted)
    wav_path = WAV_DIR / f'{vid}.wav'
    if wav_path.exists():
        return str(wav_path)
    
    # Try original locations
    for folder in ['chuckle_audio', 'chuckle_audio_all/audio', 
                  'chuckle_audio_all/audio_final', 'chuckle_audio_all/audio_new']:
        audio_dir = BASE / folder
        if audio_dir.exists():
            for ext in ['.wav', '.mp3', '.m4a']:
                p = audio_dir / f'{vid}{ext}'
                if p.exists() and not p.name.endswith('.part'):
                    return str(p)
    return None

In [ ]:
# Extract prosody for all videos
print('🔍 Extracting prosody for all videos...')
t0 = time.time()

prosody_data = {}  # video_id -> list of prosody arrays
failed = []

for i, vid in enumerate(wavlm_data.keys()):
    audio_path = get_audio_path(vid)
    
    if not audio_path:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(wavlm_data[vid])
        continue
    
    try:
        # Load audio
        if audio_path.endswith('.wav'):
            y, sr = sf.read(audio_path, dtype='float32')
        else:
            y, sr = librosa.load(audio_path, sr=SR, mono=True)
        
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        
        if sr != SR:
            y = librosa.resample(y, orig_sr=sr, target_sr=SR)
        
        # Extract prosody for each utterance
        video_prosody = []
        for emb in wavlm_data[vid]:
            start_s = emb['start']
            end_s = emb['end']
            
            start_sample = int(start_s * SR)
            end_sample = int(end_s * SR)
            
            if end_sample > len(y):
                end_sample = len(y)
            
            y_slice = y[start_sample:end_sample]
            
            if len(y_slice) < SR * 0.1:
                video_prosody.append(np.zeros(21, dtype=np.float32).tolist())
            else:
                prosody = extract_prosody_21dim(y_slice, SR)
                video_prosody.append(prosody.tolist())
        
        prosody_data[vid] = video_prosody
        
    except Exception as e:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(wavlm_data[vid])
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(wavlm_data) - i - 1)
        print(f'📊 {i+1}/{len(wavlm_data)} | ETA: {eta/60:.1f} min | Failed: {len(failed)}')

print(f'\n✅ Prosody extracted for {len(prosody_data)} videos')
print(f'   Failed (no audio): {len(failed)}')
print(f'   Time: {(time.time()-t0)/60:.1f} min')

## Step 10: Combine features and train

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

class FusionDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        d = self.data[idx]
        wavlm = torch.tensor(d['wavlm'], dtype=torch.float32)
        prosody = torch.tensor(d['prosody'], dtype=torch.float32)
        x = torch.cat([wavlm, prosody])  # 768 + 21 = 789
        y = torch.tensor(d['label'], dtype=torch.float32)
        return x, y

# Prepare combined data
def prepare_data(data_list, prosody_dict):
    result = []
    for d in data_list:
        vid = d.get('video_id') or d.get('video_id')
        if not vid:
            # Match by start/end
            for v in wavlm_data:
                for i, emb in enumerate(wavlm_data[v]):
                    if abs(emb['start'] - d['start']) < 0.01 and abs(emb['end'] - d['end']) < 0.01:
                        vid = v
                        break
        
        if vid in prosody_dict:
            # Find index
            for i, emb in enumerate(wavlm_data.get(vid, [])):
                if abs(emb['start'] - d['start']) < 0.01 and abs(emb['end'] - d['end']) < 0.01:
                    result.append({
                        'wavlm': emb['embedding'],
                        'prosody': prosody_dict[vid][i],
                        'label': d['label']
                    })
                    break
    return result

print('📦 Preparing combined datasets...')
train_ds_data = prepare_data(train_data, prosody_data)
val_ds_data = prepare_data(val_data, prosody_data)
test_ds_data = prepare_data(test_data, prosody_data)

train_ds = FusionDataset(train_ds_data)
val_ds = FusionDataset(val_ds_data)
test_ds = FusionDataset(test_ds_data)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
test_loader = DataLoader(test_ds, batch_size=64)

print(f'✅ Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')

In [ ]:
import torch.nn as nn

class FusionModel(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(wavlm_dim + prosody_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1),
        )
    
    def forward(self, x):
        return self.net(x).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️ Device: {device}')

model = FusionModel().to(device)

# Class weights
all_labels = [d['label'] for d in train_ds_data]
pos_weight = compute_class_weight('balanced', classes=[0,1], y=all_labels)[1]
pos_weight = torch.tensor([pos_weight]).to(device)
print(f'⚖️ Pos weight: {pos_weight.item():.2f}')

criterion = nn.BCELoss(pos_weight=pos_weight)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

EPOCHS = 10
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device)
            out = model(x)
            val_preds.extend((out > 0.5).cpu().numpy())
            val_labels.extend(y.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, str(BASE / 'best_fusion_model_v15.pt'))
        print(f'  ✅ New best!')

print(f'\n🏆 Best Val F1: {best_f1:.4f}')

In [ ]:
# Final evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device)
        out = model(x)
        test_preds.extend((out > 0.5).cpu().numpy())
        test_labels.extend(y.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'🏆 Test F1: {test_f1:.4f}')
print()
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))